# Ancestry filtering: AoU's premade PCs

Filters directly on AoU's own genomic ancestry PCA (`ancestry_preds.tsv`'s `pca_features`, 16 PCs) instead of building a from-scratch 1000G-projection pipeline. Replaces `01_build_1000g_reference.ipynb` / `02_build_ancestry_panel_hm3.ipynb` / `03_round2_1000g_filter.ipynb` entirely -- no ID+REF+ALT harmonization, no dsub/Batch, no variant-level QC. AoU already computed the PC space; we just filter on it.

`training_pca.tsv` (AoU's own reference panel, HGDP+1000G-style, ~4,151 samples) sits in the *same* 16-PC space as `pca_features` -- no projection needed to orient AoU samples relative to reference populations, just plot both on the same axes.

Approach: Mahalanobis distance from each AoU sample to a reference population's centroid (mean + covariance of `training_pca.tsv`'s matching `pop_label` rows), gated at different quantile thresholds. EUR gets three widths (`eur_strict`/`eur_base`/`eur_loose`); AFR and EAS each get one threshold. A separate `uniform` sample set is pool-wide (every AoU sample, no ancestry gate) and density-normalized across PC space instead of Mahalanobis-gated.

## Inputs

In [ ]:
import os
import ast
import numpy as np
import pandas as pd
from scipy.stats import chi2
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
CDR_VERSION = "v9"

# Top-level bucket folder name for this project's outputs -- distinct from
# CDR_VERSION, which keeps its real meaning elsewhere. Fixed literal, matches
# every other notebook in this pipeline.
PROJECT_DIR = "phenotypic_covariance_v9"

ANCESTRY_BUCKET_DIR = f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/01_ancestry_filtering"
FINAL_PCA_DIR = f"{ANCESTRY_BUCKET_DIR}/ancestry_pca_filter/final_pca"
os.makedirs(FINAL_PCA_DIR, exist_ok=True)

# confirmed via a real `ls` of the mounted v9 CDR
ANCESTRY_AUX_DIR = os.path.expanduser(
    "~/workspace/cdrv9/vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/aux/ancestry"
)
ANCESTRY_PREDS_PATH = os.path.join(ANCESTRY_AUX_DIR, "ancestry_preds.tsv")
TRAINING_PCA_PATH = os.path.join(ANCESTRY_AUX_DIR, "training_pca.tsv")
assert os.path.isfile(ANCESTRY_PREDS_PATH), f"missing {ANCESTRY_PREDS_PATH!r}"
assert os.path.isfile(TRAINING_PCA_PATH), f"missing {TRAINING_PCA_PATH!r}"

N_PCS_TOTAL = 16   # confirmed via eigenvalues.txt (16 rows) and a real pca_features/scores row

print(FINAL_PCA_DIR)
print(ANCESTRY_PREDS_PATH)
print(TRAINING_PCA_PATH)

## Load AoU samples + reference panel

Both `pca_features` and `scores` are Python-literal bracketed lists of 16 floats -- `ast.literal_eval`, no custom parsing needed.

In [ ]:
def parse_pc_column(series, n_pcs=N_PCS_TOTAL):
    parsed = series.apply(ast.literal_eval)
    arr = np.vstack(parsed.values)
    assert arr.shape[1] == n_pcs, f"expected {n_pcs} PCs, got {arr.shape[1]}"
    return pd.DataFrame(arr, columns=[f"PC{i}" for i in range(1, n_pcs + 1)], index=series.index)

aou = pd.read_csv(ANCESTRY_PREDS_PATH, sep="\t")
aou_pcs = parse_pc_column(aou["pca_features"])
aou = pd.concat([aou[["research_id", "ancestry_pred"]], aou_pcs], axis=1)
aou = aou.rename(columns={"research_id": "person_id"})

ref = pd.read_csv(TRAINING_PCA_PATH, sep="\t")
ref_pcs = parse_pc_column(ref["scores"])
ref = pd.concat([ref[["s", "pop_label", "project_meta.project_pop"]], ref_pcs], axis=1)
ref = ref.rename(columns={"s": "sample", "project_meta.project_pop": "project_pop"})

print(f"{len(aou)} AoU samples, {len(ref)} reference samples")
print("AoU ancestry_pred counts:")
print(aou["ancestry_pred"].value_counts())
print("Reference pop_label counts:")
print(ref["pop_label"].value_counts())

## Standardized-Euclidean gate

Z-score each PC by the reference population's own mean/SD, then plain Euclidean distance from that group's centroid in z-units -- equivalent to Mahalanobis distance with a diagonal covariance (correlations between PCs within a group ignored). Chosen over the full-covariance Mahalanobis ellipsoid after comparing both: the diagonal version's circular gate in standardized space gave cleaner, more interpretable retained-count behavior across thresholds. Mean, not mode -- reference `pop_label` clusters are expected to be roughly unimodal.

In [ ]:
def euclid_z(x, mean, std):
    z = (x - mean) / std
    return np.sqrt(np.sum(z ** 2))

def fit_group(ref_df, group, n_pcs):
    pc_cols = [f"PC{i}" for i in range(1, n_pcs + 1)]
    group_pcs = ref_df.loc[ref_df["pop_label"] == group, pc_cols].values
    assert len(group_pcs) > n_pcs, f"too few reference samples ({len(group_pcs)}) for {group!r}"
    mean = group_pcs.mean(axis=0)
    std = group_pcs.std(axis=0, ddof=1)
    return mean, std

def gate(aou_df, mean, std, threshold_quantile, n_pcs):
    pc_cols = [f"PC{i}" for i in range(1, n_pcs + 1)]
    threshold = np.sqrt(chi2.ppf(threshold_quantile, df=n_pcs))
    dists = np.array([euclid_z(row, mean, std) for row in aou_df[pc_cols].values])
    return dists, dists <= threshold

## Compare PCs 1-5 vs PCs 1-2

Fit each group's centroid/covariance both ways, compare retained counts at the same threshold quantile before committing to one `n_pcs`. Default assumption is 5 -- more PCs should resolve finer sub-structure (e.g. within-EUR), but only if those PCs actually carry population signal rather than noise for these groups.

In [ ]:
GROUPS = ["eur", "afr", "eas"]
COMPARE_THRESHOLD = 0.999   # arbitrary fixed quantile, just for the n_pcs comparison

for n_pcs in (5, 2):
    print(f"--- n_pcs={n_pcs} ---")
    for group in GROUPS:
        mean, std = fit_group(ref, group, n_pcs)
        _, keep_mask = gate(aou, mean, std, COMPARE_THRESHOLD, n_pcs)
        print(f"  {group}: {keep_mask.sum()} / {len(aou)} retained at q={COMPARE_THRESHOLD}")

## Pick `N_PCS`

Set after inspecting the comparison above.

In [ ]:
N_PCS = 5   # <-- set after reviewing apf-npcs-compare's output above

## Sample sets

EUR gets three widths; AFR and EAS each get one threshold. `prob_tag` mirrors the earlier pipeline's convention (`f"p{{threshold*100:g}}"`) so downstream notebooks' file-naming pattern is unchanged.

In [ ]:
SAMPLE_SETS = {
    "eur_strict": {"group": "eur", "threshold": 0.1},
    "eur_base":   {"group": "eur", "threshold": 0.5},
    "eur_loose":  {"group": "eur", "threshold": 0.99},
    "afr":        {"group": "afr", "threshold": 0.02},
    "eas":        {"group": "eas", "threshold": 0.9},
}

def prob_tag(threshold):
    return f"p{threshold * 100:g}"

for cfg in SAMPLE_SETS.values():
    cfg["prob_tag"] = prob_tag(cfg["threshold"])

print(SAMPLE_SETS)

## Run the gate, write keep-lists

In [ ]:
keep_masks = {}
group_fits = {group: fit_group(ref, group, N_PCS) for group in GROUPS}

for sample_set, cfg in SAMPLE_SETS.items():
    mean, std = group_fits[cfg["group"]]
    dists, keep_mask = gate(aou, mean, std, cfg["threshold"], N_PCS)
    keep_masks[sample_set] = keep_mask

    sample_set_dir = os.path.join(FINAL_PCA_DIR, sample_set)
    os.makedirs(sample_set_dir, exist_ok=True)
    keep_ids = aou.loc[keep_mask, "person_id"]
    out_path = os.path.join(sample_set_dir, f"final_keep_ids_{sample_set}_{cfg['prob_tag']}.txt")
    keep_ids.to_csv(out_path, index=False, header=False)
    print(f"[{sample_set}] {keep_mask.sum()} / {len(aou)} kept -> {out_path}")

## `uniform`: population-wide, density-normalized sample

Pool-wide, not ancestry-gated -- every AoU sample, no Mahalanobis cut. Natural sampling density in PC space is highest near dense clusters (e.g. the EUR-heavy region) and thins elsewhere; this can bias downstream heritability/GRM estimates toward whatever's densest rather than reflecting genetic diversity evenly. Same design as Steiner et al. 2024 (PNAS, *"Study design and the sampling of deleterious rare variants in biobank-scale datasets"*)'s uniform-vs-Gaussian PC-space sampling comparison in the UK Biobank: draw target points uniformly at random over the PC1-PC2 bounding box, then match each to its nearest not-yet-used real sample (nearest-neighbor matching without replacement). This directly normalizes for density -- dense regions contribute samples roughly proportional to their *area* in PC space, not their point count, since target points are area-uniform.

Restricted to PC1-PC2 (matching Steiner et al.'s own choice) -- nearest-neighbor matching against a uniform grid/random draw only stays meaningful at low dimension; higher-dimensional bounding boxes are mostly empty space (curse of dimensionality), so most target draws would have no nearby real sample.

In [ ]:
UNIFORM_TARGET_N = 100_000   # arbitrary fixed target size

xy_all = aou[["PC1", "PC2"]].values
pc1_lo, pc1_hi = np.quantile(xy_all[:, 0], [0.00001, 0.99999])   # trim extreme tails so most target
pc2_lo, pc2_hi = np.quantile(xy_all[:, 1], [0.00001, 0.99999])   # draws land near real data, not empty space

tree = cKDTree(xy_all)
rng = np.random.default_rng(0)

n_target = min(UNIFORM_TARGET_N, len(aou))
used = np.zeros(len(aou), dtype=bool)
matched_idx = []

# oversample target draws since some will collide with an already-used nearest
# neighbor and need a retry -- draw in batches until n_target unique matches found
batch_size = max(1000, n_target // 2)
while len(matched_idx) < n_target:
    draw_x = rng.uniform(pc1_lo, pc1_hi, size=batch_size)
    draw_y = rng.uniform(pc2_lo, pc2_hi, size=batch_size)
    targets = np.column_stack([draw_x, draw_y])

    # k=8 nearest candidates per target, so a used first-choice still has fallbacks
    _, nn_idx = tree.query(targets, k=8)
    for row in nn_idx:
        for candidate in row:
            if not used[candidate]:
                used[candidate] = True
                matched_idx.append(candidate)
                break
        if len(matched_idx) >= n_target:
            break

matched_idx = np.array(matched_idx[:n_target])
uniform_df = aou.iloc[matched_idx]

uniform_dir = os.path.join(FINAL_PCA_DIR, "uniform")
os.makedirs(uniform_dir, exist_ok=True)
out_path = os.path.join(uniform_dir, "final_keep_ids_uniform_uniform.txt")
uniform_df["person_id"].to_csv(out_path, index=False, header=False)
print(f"[uniform] {len(uniform_df)} / {len(aou)} (population-wide) -> {out_path}")

# hist2d, not a scatter -- density is exactly the thing being checked here, and a
# scatter of 100k+ points just looks like a solid blob at any reasonable point size
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
axes[0].hist2d(xy_all[:, 0], xy_all[:, 1], cmap="PuRd", bins=200)
axes[0].set_title(f"all AoU (n={len(aou)})")
axes[1].hist2d(uniform_df["PC1"], uniform_df["PC2"], cmap="PuRd", bins=200)
axes[1].set_title(f"uniform (n={len(uniform_df)})")
for ax in axes:
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
plt.tight_layout()
plot_path = os.path.join(uniform_dir, "uniform_vs_all.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")

## Orientation plot (not a filtering step)

AoU samples (downsampled for plotting speed) and `training_pca.tsv`'s reference points on the same PC1/PC2 axes -- both already share this exact PC space, so no projection is needed. Purely a visual check that each gate lands where expected relative to the reference clusters.

In [ ]:
PLOT_N_AOU = 50_000

# hist2d background instead of a faint gray scatter -- a light colormap density
# map stays visible under the reference/sample_set overlays instead of getting
# washed out by them (the earlier scatter version was hard to see once the
# reference "x" markers and sample_set dots were drawn on top)
fig, ax = plt.subplots(figsize=(9, 8))
ax.hist2d(aou["PC1"], aou["PC2"], bins=250, cmap="Greys", norm=plt.matplotlib.colors.LogNorm())

for sample_set in SAMPLE_SETS:
    mask = keep_masks[sample_set]
    sub = aou.loc[mask].sample(n=min(PLOT_N_AOU, mask.sum()), random_state=0)
    ax.scatter(sub["PC1"], sub["PC2"], s=3, alpha=0.5, label=sample_set)

for group in GROUPS:
    sub = ref[ref["pop_label"] == group]
    ax.scatter(sub["PC1"], sub["PC2"], s=14, marker="x", linewidths=0.8, label=f"ref: {group}")

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("AoU samples (density) vs. training_pca.tsv reference populations")
ax.legend(fontsize=7, markerscale=2, loc="best")
plt.tight_layout()
plot_path = os.path.join(FINAL_PCA_DIR, "orientation_plot.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")

## Deeper PCs, per sample set

One row per `SAMPLE_SET` (+ `uniform`), walking PC1 through PC10 as consecutive pairs (PC1v2, PC3v4, ..., PC9v10) rather than a full 45-panel pairwise grid. Each row shows only its own gate boundary and its own gated points -- density background from all of AoU underneath.

Gate boundary is drawn as a proper ellipse (`width`/`height` independently scaled by each PC's own SD, `df=N_PCS` matching what the real gate used) -- it's only a tight bound in the PC1v2 panel; for `N_PCS>2` a point can sit inside a later panel's ellipse while still being excluded by distance from PCs not shown in that panel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from matplotlib.colors import LogNorm

# refit each group's mean/std over 10 PCs specifically for this plot -- group_fits
# from apf-gate-run was built with N_PCS, which may not reach PC9/PC10
PLOT_N_PCS = 10
group_fits_10 = {group: fit_group(ref, group, PLOT_N_PCS) for group in GROUPS}

# consecutive PC pairs, not a full pairwise grid (10 choose 2 = 45 panels is too
# many to read) -- standard "biplot ladder" convention for checking whether
# structure shows up beyond PC1/PC2
PC_PAIRS = [(1, 2), (3, 4), (5, 6), (7, 8), (9, 10)]

GROUP_COLORS = {"eur": "tab:blue", "afr": "tab:orange", "eas": "tab:green"}
PLOT_N_PER_SET = 3_000   # downsample each sample_set for scatter -- full N just re-blobs the panel

# one row per sample_set (+ uniform, no single group of its own) -- avoids
# stacking every sample_set's points/circle into the same panel
ROWS = list(SAMPLE_SETS.items()) + ([("uniform", None)] if "uniform_df" in dir() else [])

fig, axes = plt.subplots(len(ROWS), len(PC_PAIRS), figsize=(4.5 * len(PC_PAIRS), 4 * len(ROWS)))
if len(ROWS) == 1:
    axes = axes.reshape(1, -1)

for row, (sample_set, cfg) in enumerate(ROWS):
    for col, (i, j) in enumerate(PC_PAIRS):
        ax = axes[row, col]
        pc_i, pc_j = f"PC{i}", f"PC{j}"

        # density background -- stays visible under overlays, unlike a raw scatter
        ax.hist2d(aou[pc_i], aou[pc_j], bins=200, cmap="Greys", norm=LogNorm())

        if sample_set == "uniform":
            sub = uniform_df.sample(n=min(PLOT_N_PER_SET, len(uniform_df)), random_state=0)
            ax.scatter(sub[pc_i], sub[pc_j], s=3, alpha=0.35, color="black")
        else:
            mean, std = group_fits_10[cfg["group"]]
            # gate is standardized-Euclidean over N_PCS dims (df=N_PCS, NOT
            # PLOT_N_PCS -- the real gate never saw PC6-10 if N_PCS<10) -- the
            # boundary is a circle in z-space, but back in raw PC units it's an
            # axis-aligned ELLIPSE whenever std[i-1] != std[j-1] (always, in
            # practice, since PC variance drops PC-over-PC). matplotlib's Circle
            # only takes one radius -- Ellipse is required to actually match.
            radius = np.sqrt(chi2.ppf(cfg["threshold"], df=N_PCS))
            ax.add_patch(Ellipse(
                (mean[i - 1], mean[j - 1]),
                width=2 * radius * std[i - 1],
                height=2 * radius * std[j - 1],
                fill=False, edgecolor=GROUP_COLORS[cfg["group"]], linewidth=1.4,
            ))

            mask = keep_masks[sample_set]
            sub = aou.loc[mask].sample(n=min(PLOT_N_PER_SET, mask.sum()), random_state=0)
            ax.scatter(sub[pc_i], sub[pc_j], s=3, alpha=0.35, color=GROUP_COLORS[cfg["group"]])

        if row == 0:
            ax.set_title(f"{pc_i} vs {pc_j}")
        ax.set_ylabel(f"{sample_set}\n{pc_j}" if col == 0 else pc_j)
        ax.set_xlabel(pc_i)

plt.tight_layout()
plot_path = os.path.join(FINAL_PCA_DIR, "deeper_pcs_per_sample_set.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")

## Next steps

`02_genome_wide_qc_thinning_batch_submit.ipynb` reads these keep-lists next, one `SAMPLE_SET` at a time, to build the GRM panel.